# ⛷️ Homework 6: Gradient Descent

## Due Date: Friday, October 23rd, 11:59 PM
You must submit this assignment to Pensive by the on-time deadline, Friday, October 23rd, 11:59 PM. Please read the syllabus for the Slip Day policy. No late submissions beyond what is outlined in the Slip Day policy will be accepted. **We strongly encourage you to submit your work to Pensive several hours before the stated deadline.** This way, you will have ample time to reach out to staff for support if you encounter difficulties with submission. While course staff is happy to help guide you with submitting your assignment ahead of the deadline, we will not respond to last-minute requests for assistance.

Please read the instructions carefully when submitting your work to Pensive. 

## 💯 Score Breakdown

Question | Manual | Points
--- |---| ---
0 | No | 1                                                                         
1ai | Yes | 1
1aii | No |3
1b | No | 3
2a | Yes | 1
2b | Yes | 2
2c | Yes | 2
2di | Yes | 2
2dii | Yes |1
2diii | Yes | 3
2div | Yes | 2
2dv | Yes | 1
2dvi | Yes | 3
2dvii | Yes | 1
3a | Yes | 3
3b | Yes | 1                                                                                                                
Total | | 30

## 🏎️ Before You Start

For each question in the assignment, please write down your answer in the answer cell(s) right below the question.

**For questions marked "[ON PAPER]," please ensure you handwrite your responses on either paper or a tablet!** You'll submit these to the "Homework 6 Handwritten Work" assignment on Pensive.    

We understand that it is helpful to have extra cells breaking down the process towards reaching your final answer. If you happen to create new cells below your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It will cause errors when we run the autograder, and it will sometimes cause a failure to generate the PDF file.

**For this assignment, you should not use LLMs to help you complete the questions, except for the few subparts that explicitly encourage you to do so.**

## Debugging Guide
If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/). In this guide, you can find general questions about Jupyter notebooks / Datahub, Pensive, and common `Polars` errors.

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw06.ipynb")

## Collaboration Policy

Data science is a collaborative activity. While you may talk to others about assignments, we ask that you **write your solutions individually**. If you do discuss the assignments with others, please **include their names** in the following cell:

**Collaborators**: *list names here*

In [ ]:
# Run this cell to set up your notebook.
import csv
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import zipfile
from pathlib import Path

# Default plot configurations
%matplotlib inline
plt.rcParams['figure.figsize'] = (16,8)
plt.rcParams['figure.dpi'] = 150
sns.set()

import warnings
warnings.filterwarnings("ignore")

from IPython.display import display, Latex, Markdown

<br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Question 0: Mid-Semester Survey
Please fill out the anonymous mid-semester survey. You will have received a link to this form from
data100.instructors@berkeley.edu by Sunday, 10/18 @ 11:59 PM. Please reach out to course staff if you have not received the link after
 this time. Please do not share your emailed link with other students.
At the end of the survey, you'll receive a password. Assign qsurvey to the password to receive credit for completing the
survey. **Don't forget to assign the password as a string!**

In [ ]:
qsurvey = ...

In [ ]:
grader.check("q0")

## Question 1: Introduction to Gradient Descent

In [ ]:
# Run this cell to load the data for this problem; no further action is needed.
df = pl.read_csv("data/hw6_data.csv").drop("")
df.head()

If we plot this data, we see that there is a clear sinusoidal relationship between `x` and `y`. Here we use `plotly` to plot a scatterplot showing the relationship between `x` and `y`. If you hover over points, you will be able to see the details of the data that's attributed to a data point.

In [ ]:
# Run this cell to plot the data; no further action is needed.
import plotly.express as px
px.scatter(df, x="x", y="y", width=800, height=600)

For question 1, we're going to model the relationship of our data by:

$$\Large{
\hat{y}_\theta(x) = \theta_1x + \sin(\theta_2x)
}$$

Our model is parameterized by $\theta = \begin{bmatrix} \theta_1 \\ \theta_2 \end{bmatrix}$.

Note that a general `sin` function $a\sin(bx+c)$ has three parameters: amplitude scaling parameter $a$, frequency parameter $b$, and phase shifting parameter $c$. 
Here, we're assuming the amplitude $a$ is around 1, and the phase shifting parameter $c$ is around zero. 

We define the `sin_model` function below that outputs predictions $\hat{\mathbb{Y}}$ for the $y$-values $\mathbb{Y}$ using $\mathbb{X}$ (the $x$-values) based on our new equation.

In [ ]:
# Run this cell to define the model; no further action is needed.
def sin_model(X, theta):
    """
    Outputs predictions (Y-hat) given X, theta_1, theta_2.

    Keyword arguments:
    x -- The vector of values x.
    theta -- A vector of length 2, where theta[0] = theta_1 and theta[1] = theta_2.
    """
    theta_1 = theta[0]
    theta_2 = theta[1]
    return theta_1 * X + np.sin(theta_2 * X)

### Terminology: Loss

We use the word "loss" in two different (but very related) contexts in this course.
* In general, the loss is the cost function that measures how far off a model's prediction(s) is(are) from the actual value(s).
    * **Per-datapoint loss** is a cost function that measures the cost of $y$ vs $\hat{y}$ for a particular datapoint. For example, $L(y, \hat{y}) = (y - \hat{y})^2$ is the L2 loss of the observed and predicted pair ($y$, $\hat{y}$).
    * **Loss** (without any adjectives) is generally a cost function measured across all data points. We often use this term interchangeably with **empirical risk** to denote the average per-datapoint loss. For example, MSE can be denoted as $MSE(\theta) = L(\theta) = R(\theta) = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y_i})^2$
* If we're being particular about notation, we usually prioritize using the latter definition, because we don't particularly look at a given data point's loss when optimizing a model. In other words, the dataset-level loss is the **objective function** that we'd like to minimize.
    * Example: "gradient of L2 loss" means gradient of Mean Squared Error, not per-datapoint L2 loss.

<br>

---
### Question 1a

Recall $\hat{\theta}$ is the value of $\theta$ that minimizes our loss function. One way to solve for $\hat{\theta}$ is by computing the gradient of our loss function with respect to $\theta$, like we have seen in lecture. Recall that the gradient is a column vector of partial derivatives (one partial derivative per parameter).

*As a reminder, our model for question 1 is* ${
\hat{y}_\theta(x) = \theta_1x + \sin(\theta_2x)
}$.

#### [ON PAPER] Question 1ai

Write/derive the expressions for the following values. Working out the partial derivatives on paper will be a great starting point for implementing `sin_MSE_dt1` and `sin_MSE_dt2`.

* $L(\theta)$ as `sin_mse`: Our cost function, mean squared error, where `theta` represents $\theta = \begin{bmatrix} \theta_1 \\ \theta_2 \end{bmatrix}$. 
Recall that $$L(\vec{\theta}, \mathbb{X}, \mathbb{Y}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2.$$
* $\frac{\partial L }{\partial \theta_1}$ as `sin_MSE_dt1`: The partial derivative of $L$ with respect to $\theta_1$.
* $\frac{\partial L }{\partial \theta_2}$ as `sin_MSE_dt2`: The partial derivative of $L$ with respect to $\theta_2$.

**Please handwrite your response to this question on paper or a tablet.** Remember that you'll submit this, along with your other handwritten responses, to the **Homework 6 Handwritten Work** assignment on Pensive.

*Hint: this will be easier if you avoid simplifying or expanding as much as possible. Instead, rely on the chain rule when taking derivatives.*

#### Question 1aii
Now fill in the functions below using your work from 1ai. Note that `sin_MSE_gradient` computes $\nabla_{\vec{\theta}} L(\vec{\theta}, \mathbb{X}, \mathbb{Y})$  by calling `sin_MSE_dt1` and `sin_MSE_dt2` and returns the gradient `dt`.

**Notes:**
* We are still working with the `DataFrame` `df`.
* To keep our code more concise, call `.mean()` on the result instead of taking `.sum()` and then dividing by the number of data points.
* Another way to keep your code more concise is to use the function `sin_model` we defined which computes the output of the model.
* Review the "Terminology: Loss" section above question 1a.
* All vectors should be represented as NumPy arrays with two elements.

In [ ]:
def sin_MSE(theta, X, Y):
    """
    Returns the numerical value of the l2 loss of our sinusoidal model given theta.

    Keyword arguments:
    theta -- The vector of values theta.
    X     -- The vector of x values - note that sin_model only needs a vector of X values and handles the transformations.
    Y     -- The vector of y values.
    """
    ...

def sin_MSE_dt1(theta, X, Y):
    """
    Returns the numerical value of the partial derivative of l2 loss with respect to theta_1.

    Keyword arguments:
    theta -- The vector of values theta.
    X     -- The vector of x values.
    Y     -- The vector of y values.
    """
    ...
    
def sin_MSE_dt2(theta, X, Y):
    """
    Returns the numerical value of the partial of l2 loss with respect to theta_2.

    Keyword arguments:
    theta -- The vector of values theta.
    X     -- The vector of x values.
    Y     -- The vector of y values.
    """
    ...
    
def sin_MSE_gradient(theta, X, Y):
    """
    Returns an array containing the gradient of l2 loss with respect to vector theta. 

    Keyword arguments:
    theta -- The vector of values theta.
    X     -- The vector of x values.
    Y     -- The vector of y values.
    """
    ...

In [ ]:
grader.check("q1aii")

<br>

---
### Question 1b

Let's now implement gradient descent. 

Recall that the gradient descent update function follows the form:

$$\large
\vec{\theta}^{(t+1)} \leftarrow \vec{\theta}^{(t)} - \alpha \left (\nabla_{\vec{\theta}} L(\vec{\theta}, \mathbb{X}, \mathbb{Y}) \right )
$$
where 

- $\vec{\theta}^{(t+1)}$: Our new coefficients after making an update.
- $\vec{\theta}^{(t)}$: Our current coefficients.
- $\alpha$: Our step size or learning rate.
- $\nabla_{\vec{\theta}} L(\vec{\theta}, \mathbb{X}, \mathbb{Y})$: The gradient of our loss.

At each time step, use the gradient and `alpha` to update your current `theta`. Save the current `theta` in `theta_history`, along with the average squared loss (computed with the current `theta`) in `loss_history`.

After completing the function, the cell will output the trajectory from running gradient descent over time.


Note that the function you're implementing here is somewhat different than the gradient descent function we created in lecture. The version in lecture was `gradient_descent(deriv_f, initial_guess, alpha, n)`, where `deriv_f` was the gradient of the function we are minimizing and `initial_guess` are the starting parameters for that function. Here our signature is a bit different (described below in docstring) than the `gradient_descent` implementation from the lecture.

**Hints:**
- Write out the gradient update equation (1 step). What variables will you need for each gradient update? Of these variables, which ones do you already have, and which ones will you need to recompute at each time step?
- You may need a loop here to update `theta` several times. Take a look at `num_iter`.
- Be sure to include the initial theta and loss into the trajectory because the test checks for this.
- Don't forget that `sin_MSE` and `sin_MSE_gradient` require the $x$ and $y$ values to be supplied.

In [ ]:
def init_theta():
    """Creates an initial theta [0., 0.] of shape (2,) as a starting point for gradient descent."""
    return np.array([0.0, 0.0])

def grad_desc(loss_f, gradient_loss_f, theta, data, num_iter=20, alpha=0.1):
    """
    Run gradient descent update for a finite number of iterations and static learning rate.

    Keyword arguments:
    loss_f -- The loss function to be minimized (used for computing loss_history).
    gradient_loss_f -- The gradient of the loss function to be minimized.
    theta -- The vector of values theta to use at the first iteration.
    data -- The data used in the model. 
    num_iter -- The max number of iterations.
    alpha -- The learning rate (also called the step size).
    
    Return:
    theta -- The optimal value of theta after num_iter of gradient descent.
    theta_history -- The list of theta values over each iteration of gradient descent.
    loss_history -- The list of loss values over each iteration of gradient descent.
    """
    theta_history = []
    loss_history = []

    ...
    return theta, theta_history, loss_history

theta_start = init_theta()
theta_hat, thetas_used, losses_calculated = grad_desc(
    sin_MSE, sin_MSE_gradient, theta_start, df, num_iter=20, alpha=0.1
)
for b, l in zip(thetas_used, losses_calculated):
    print(f"theta: {b}, Loss: {l}")

In [ ]:
grader.check("q1b")

Let's visually inspect our results of running gradient descent to optimize $\vec{\theta}$. The code below plots our $x$-values with our model's predicted $\hat{y}$-values over the original scatter plot. You should notice that gradient descent successfully optimized $\vec{\theta}$.

In [ ]:
theta_init = init_theta()
theta_est, thetas, loss = grad_desc(sin_MSE, sin_MSE_gradient, theta_init, df)

Plotting our model output over our observations shows that gradient descent did  a great job finding both the overall increase (slope) of the data, as well as the oscillation frequency.

In [ ]:
X, Y = df['x'], df['y']
Y_pred = sin_model(X, theta_est)

plt.plot(X, Y_pred, label='Model ($\hat{y}$)')
plt.scatter(X, Y, alpha=0.5, label='Observation ($y$)', color='gold')
plt.legend();

## Visualizing Loss

Let's visualize our loss functions and gain some insight as to how gradient descent optimizes our model parameters. In this part, we'll see the trajectory of the algorithm as it travels the loss surface. Run the following cells to see a visualization of this trajectory.

In [ ]:
thetas = np.array(thetas).squeeze()
loss = np.array(loss)
thetas

In [ ]:
# Run this cell to see a 3D plot (gradient descent with static alpha).
from hw6_utils import plot_3d
from sklearn.metrics import mean_squared_error
plot_3d(thetas[:, 0], thetas[:, 1], loss, mean_squared_error, sin_model, X, Y)

In [ ]:
def contour_plot(title, theta_history, loss_function, model, X, Y):
    """
    The function takes the following as argument:
        title: name of the plot.
        theta_history: An (N, 2) array of theta history.
        loss_function: For example, l2_loss.
        model: For example, sin_model.
        X: The original X input.
        Y: The original Y output.
    """
    theta_1_series = theta_history[:,0] # A list or array of theta_1 value
    theta_2_series = theta_history[:,1] # A list or array of theta_2 value

    ## In the following block of code, we generate the z-value
    ## across a 2D grid
    theta1_s = np.linspace(np.min(theta_1_series) - 0.1, np.max(theta_1_series) + 0.1)
    theta2_s = np.linspace(np.min(theta_2_series) - 0.1, np.max(theta_2_series) + 0.1)

    x_s, y_s = np.meshgrid(theta1_s, theta2_s)
    data = np.stack([x_s.flatten(), y_s.flatten()]).T
    ls = []
    for theta1, theta2 in data:
        l = loss_function(model(X, np.array([theta1, theta2])), Y)
        ls.append(l)
    z = np.array(ls).reshape(50, 50)

    fig1, ax = plt.subplots(figsize=(30, 10))
    cs = ax.contourf(x_s, y_s, z, levels=20, cmap='viridis')
    ax.scatter(theta_1_series, theta_2_series, c='w')
    ax.plot(theta_1_series, theta_2_series, c='w')

    cbar = fig1.colorbar(cs)

    plt.title(title)

In [ ]:
contour_plot('Gradient Descent with Static Learning Rate', thetas, mean_squared_error, sin_model, df["x"], df["y"])

As we can see, gradient descent is able to navigate even this fairly complex loss space and find a nice minimum.

<br/>
<hr style="border: 1px solid #fdb515;" />

## Question 2: Gradient Descent Practice

In this question, we'll reimplement some of the concepts discussed in question 1. This time, you'll use gradient descent to find a solution to a regression problem. **For this question, you may only use an LLM to help you with part d(i), d(iv), and d(vi): all other parts should be done by hand.** 

Consider the following model for predicting $y$ from $x$, which is slightly different from the model in question 1:

$${
\hat{y}_\theta(x) = \theta_1\sin(\theta_2x)
}$$

<!-- BEGIN QUESTION -->

<br>

---
### Question 2a

Is this model linear in $\theta_1$ and $\theta_2$? Why or why not?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---
### [ON PAPER] Question 2b

Let's use the following loss function.

$$L(\vec{\theta}, \mathbb{X}, \mathbb{Y}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2.$$

On paper or tablet, **calculate the gradient with respect to $\vec{\theta}$**. You do not need to simplify. 

*Hint: your answer should be a vector with two algebraic expressions!*

**Please handwrite your response to this question on paper or a tablet.**

<!-- BEGIN QUESTION -->

<br>

---
### Question 2c

Explain why there’s no closed-form solution for the best values of $\theta_1$ and $\theta_2$.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>

---
### Question 2d

We observe the following data points: 
$(x = 1, y = 0.5)$, $(x = 2, y = 1.8)$, $(x = 2.7, y = -0.4)$

<!-- BEGIN QUESTION -->


---
#### Question 2di

Visualize the loss surface as a heatmap (similar to the visualization from lecture 14) for a reasonable range of $\theta_1$ and $\theta_2$ values. **You may use an LLM and/or reuse code from the "Visualization Loss" section in question 1 to help you perform this task.**

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->


---
#### Question 2dii

In 1-2 sentences, explain the most important takeaways from your visualization above.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br>


For the remainder of this question, we’ll use gradient descent to find the optimizing values of $\theta_1$ and $\theta_2$. **For all remaining parts except 2div and 2dvi, you shouldn’t use any LLMs, or `sklearn` or `scipy` functions.** Make the necessary calculations with a calculator or use simple Python functions that implement your answer from part b.


---
#### [ON PAPER] Question 2diii

Suppose we initialize $\theta_1 = 3$ and $\theta_2 = -5$. Run two steps of batch gradient descent with learning rate $\alpha=0.25$. For each step, calculate the gradient, loss function evaluated at current parameters, and updated values of $\theta_1$ and $\theta_2$. Remember that we have observed the following data points: 
$(x = 1, y = 0.5)$, $(x = 2, y = 1.8)$, $(x = 2.7, y = -0.4)$

*Hint: at the end of the second step, you should get $\theta_1 = 1.227$ and $\theta_2 = -1.759$*.

**Please handwrite your response to this question on paper or a tablet.**


<!-- BEGIN QUESTION -->


---
#### Question 2div

Visualize the observed $x$, $y$ values along with the predictions from the initial model ($\theta_1 = 3$ and $\theta_2 = -5$) and the model after two steps of gradient descent ($\theta_1 = 1.227$ and $\theta_2 = -1.759$). **You may use an LLM to help you perform this task, but we strongly recommend that you try to complete this question on your own for additional practice!**

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->


---
#### Question 2dv

In 1-2 sentences, explain if or how the model got better after two steps of gradient descent.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---
#### Question 2dvi

Now suppose we accidentally increase the learning rate from $\alpha=0.25$ to $\alpha=2.5$.

Once again, visualize the loss surface as a heatmap. This time, plot the first ten gradient descent updates and initialization on your loss surface. **You may use an LLM to help you perform this task.**

*Hint: your visualization should look very similar in style to the contour plot at the end of question 1b! Feel free to reuse the* `contour_plot` *function.*

In [ ]:
X = np.array([1, 2, 2.7])
Y = np.array([0.5, 1.8, -0.4])

...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---
#### Question 2dvii
In 1-2 sentences, explain the most important takeaways from your visualization above. What was the consequence of increasing the learning rate to $\alpha = 2.5$?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>
<hr style="border: 1px solid #fdb515;" />

## Question 3: Miscellaneous GD Potpourri


---
### [ON PAPER] Question 3a

Consider three different paradigms used to train and optimize a model. Each runs for **5 epochs**. 

| Paradigm | Number of Data Points |
| :---: |:---: |
|1 (Batch GD) |      280 |
|2 (Mini-batch GD) |      140 |
|3 (Stochastic GD) |     10 |

For each of the three paradigms, compute the number of gradient computations across all epochs. Note that paradigm 2 (mini-batch GD) uses a batch size of 20.

**Please handwrite your response to this question on paper or a tablet.**


---
### [ON PAPER] Question 3b

Which of the following statements are true?

1. The initial weight(s) chosen for gradient descent can impact whether it converges to the true optimal weights.
2. The choice of loss function can impact whether gradient descent converges to the true optimal weights.
3. Both batch and stochastic gradient descent compute the true gradient of the loss surface during each iteration.

Select any of the above statements. **For each statement you select as true, provide a concrete sketch showing how it can happen, as well as a 1-2 sentence explanation of your sketch. For each statement you select as false, provide a 1-2 sentence explanation of why it is false.**

*As a reminder, the gradient descent update rule is* $\vec{\theta}^{(t+1)} \leftarrow \vec{\theta}^{(t)} - \alpha \left (\nabla_{\vec{\theta}} L(\vec{\theta}, \mathbb{X}, \mathbb{Y}) \right )$.

**Please handwrite your response to this question on paper or a tablet.**

<br/><br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Homework 06!

## Course Content Feedback

If you have any feedback about this assignment or about any of our other weekly assignments, lectures, or discussions, please fill out the [Course Content Feedback Form](https://forms.gle/3qMBFWaAmZSYhVpW9). Your input is valuable in helping us improve the quality and relevance of our content to better meet your needs and expectations!

## Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Submit this file to the Homework 6 Coding assignment on Pensive. If you run into any issues when running this cell, feel free to check this [section](https://ds100.org/debugging-guide/autograder-pensive/) in the Data 100 Debugging Guide.

**Don't forget to submit your handwritten responses to the Homework 6 Handwritten Work assignment!**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)